In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Trump Tweet Sentiment Classification

This notebook builds a complete text classification pipeline:
1. Text cleaning helper functions
2. A unified `text_cleaning_pipeline`
3. Dataset loading
4. Train-test split
5. TF-IDF vectorization
6. Logistic Regression training and evaluation

## 1. Imports and NLTK Downloads

In [2]:
import re
import pandas as pd
import numpy as np

import nltk
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## 2. Helper Functions for Text Cleaning

These mirror the functions built in the Text Preprocessing worksheet.

In [3]:
def remove_urls(text):
  """Remove URLs from text using regex."""
  url_pattern = re.compile(r'https?://\S+|www\.\S+')
  return url_pattern.sub(r'', text)


def remove_emoji(string):
  """Replace emojis in string with whitespace."""
  emoji_pattern = re.compile("["
                             u"\U0001F600-\U0001F64F"  # emoticons
                             u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                             u"\U0001F680-\U0001F6FF"  # transport & map symbols
                             u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                             u"\U00002702-\U000027B0"
                             u"\U000024C2-\U0001F251"
                             "]+", flags=re.UNICODE)
  return emoji_pattern.sub(r' ', string)


def removeunwanted_characters(document):
  """Remove @mentions, #hashtags, punctuation, emojis, and extra spaces."""
  # remove user mentions
  document = re.sub("@[A-Za-z0-9_]+", " ", document)
  # remove hashtags
  document = re.sub("#[A-Za-z0-9_]+", "", document)
  # remove punctuation / non-alphanumeric
  document = re.sub("[^0-9A-Za-z ]", "", document)
  # remove emojis
  document = remove_emoji(document)
  # collapse multiple spaces to a single space
  document = re.sub(r'\s+', ' ', document)
  return document.strip()


def lower_order(text):
  """Lowercase the text."""
  return text.lower()


# Stopwords set (built once, reused)
stop_words = set(stopwords.words('english'))
custom_stopwords = ['@', 'rt', 'amp']
stop_words.update(custom_stopwords)


def remove_stopwords(text_tokens):
  """Remove stopwords from a list of tokens."""
  return [token for token in text_tokens if token not in stop_words]


def lemmatization(token_text):
  """Lemmatize tokens (verb form)."""
  wordnet = WordNetLemmatizer()
  return [wordnet.lemmatize(token, pos='v') for token in token_text]


def stemming(token_text):
  """Apply Porter stemming to tokens."""
  porter = PorterStemmer()
  return [porter.stem(word) for word in token_text]

## 3. Text Cleaning Pipeline

Compiles all the cleaning steps into a single function. Uses **lemmatization** by default.

In [4]:
def text_cleaning_pipeline(dataset, rule="lemmatize"):
  """
  Full text cleaning pipeline.
  Input Args:
    dataset: input text string to clean.
    rule: "lemmatize" (default) or "stem" for normalization.
  Returns:
    Cleaned text as a single string with tokens joined by spaces.
  """
  # Guard against non-string input (NaN etc.)
  if not isinstance(dataset, str):
    return ""

  # Convert the input to lower order.
  data = lower_order(dataset)
  # Remove URLs
  data = remove_urls(data)
  # Remove emojis
  data = remove_emoji(data)
  # Remove all other unwanted characters (mentions, hashtags, punctuation).
  data = removeunwanted_characters(data)
  # Create tokens.
  tokens = data.split()
  # Remove stopwords
  tokens = remove_stopwords(tokens)
  # Normalize tokens
  if rule == "lemmatize":
    tokens = lemmatization(tokens)
  elif rule == "stem":
    tokens = stemming(tokens)
  else:
    print("Pick between lemmatize or stem")

  return " ".join(tokens)

In [5]:
# Quick sanity check
sample = "Hello @gabe_flomo 👋 check this out https://example.com !!! #awesome"
print(text_cleaning_pipeline(sample))

hello check


# 4. Load the Dataset

Loading `trump_tweet_sentiment_analysis.csv`. Update the path if your file lives elsewhere (e.g. in your Drive).

In [6]:
# Adjust the path as needed — e.g. '/content/drive/MyDrive/trump_tweet_sentiment_analysis.csv'
df = pd.read_csv("/content/drive/MyDrive/AI & ML/Week 08/Copy of trum_tweet_sentiment_analysis.csv", encoding="ISO-8859-1")
print("Shape:", df.shape)
df.head()

Shape: (1850123, 2)


,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [8]:
# Check columns and class distribution
print("Columns:", df.columns.tolist())
print()
print("Label distribution:")
print(df['Sentiment'].value_counts())
print()
print("Missing values:")
print(df[['text', 'Sentiment']].isna().sum())

Columns: ['text', 'Sentiment']

Label distribution:
Sentiment
0    1244211
1     605912
Name: count, dtype: int64

Missing values:
text         0
Sentiment    0
dtype: int64


In [10]:
# Drop rows where text or Sentiment is missing
df = df.dropna(subset=['text', 'Sentiment']).reset_index(drop=True)
print("After dropna:", df.shape)

After dropna: (1850123, 2)


## 5. Apply the Cleaning Pipeline

In [12]:
# Apply the cleaning pipeline to every tweet
df['cleaned_text'] = df['text'].apply(lambda t: text_cleaning_pipeline(t, rule="lemmatize"))

# Drop rows that became empty after cleaning
df = df[df['cleaned_text'].str.strip().astype(bool)].reset_index(drop=True)

print("After cleaning:", df.shape)
df[['text', 'cleaned_text', 'Sentiment']].head()

After cleaning: (1849639, 3)


,text,cleaned_text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,drain swamp taxpayer dollars trip advertise pr...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,icymi hackers rig fm radio station play antitr...,0
2,Trump protests: LGBTQ rally in New York https:...,trump protest lgbtq rally new york via,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",hi im piers morgan david beckham awful donald ...,0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,tech firm sue buzzfeed publish unverified trum...,0


## 6. Train-Test Split

In [14]:
X = df['cleaned_text']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape[0])
print("Test size :", X_test.shape[0])

Train size: 1479711
Test size : 369928


## 7. TF-IDF Vectorization

In [15]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),   # unigrams + bigrams
    min_df=2,             # ignore terms in fewer than 2 documents
    max_df=0.95,          # ignore terms in more than 95% of documents
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape :", X_test_tfidf.shape)

TF-IDF train shape: (1479711, 10000)
TF-IDF test shape : (369928, 10000)


## 8. Train Logistic Regression

In [16]:
model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='liblinear',
    random_state=42
)

model.fit(X_train_tfidf, y_train)
print("Training complete.")

Training complete.


## 9. Evaluation

In [17]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9271

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248759
           1       0.91      0.87      0.89    121169

    accuracy                           0.93    369928
   macro avg       0.92      0.91      0.92    369928
weighted avg       0.93      0.93      0.93    369928

Confusion Matrix:
[[238109  10650]
 [ 16318 104851]]


## 10. Quick Inference on New Tweets

In [18]:
def predict_sentiment(text):
  cleaned = text_cleaning_pipeline(text, rule="lemmatize")
  vec = vectorizer.transform([cleaned])
  pred = model.predict(vec)[0]
  proba = model.predict_proba(vec)[0]
  return pred, dict(zip(model.classes_, proba))


samples = [
    "Great job today, really proud of the team! #winning",
    "This is a total disaster, absolutely terrible decision.",
    "Meeting at 3pm in the main hall."
]

for s in samples:
    label, proba = predict_sentiment(s)
    print(f"Text : {s}")
    print(f"Label: {label}")
    print(f"Proba: {proba}")
    print("-" * 60)

Text : Great job today, really proud of the team! #winning
Label: 1
Proba: {np.int64(0): np.float64(0.004336083968739413), np.int64(1): np.float64(0.9956639160312606)}
------------------------------------------------------------
Text : This is a total disaster, absolutely terrible decision.
Label: 0
Proba: {np.int64(0): np.float64(0.999899884484015), np.int64(1): np.float64(0.00010011551598498072)}
------------------------------------------------------------
Text : Meeting at 3pm in the main hall.
Label: 0
Proba: {np.int64(0): np.float64(0.7125962951988645), np.int64(1): np.float64(0.2874037048011356)}
------------------------------------------------------------
